<a href="https://colab.research.google.com/github/zoesuhnny/data_science_and_ml_notes/blob/main/(smaller_dataset)_llm_notes_from_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# llm notes'
import torch
from transformers import BertTokenizer, BertModel, AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset

ds = load_dataset("google-research-datasets/poem_sentiment")

README.md:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 35.6kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.34kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.16kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/892 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/105 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/104 [00:00<?, ? examples/s]

In [2]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint) #loaded tokenizer

In [3]:
model = BertModel.from_pretrained(checkpoint) #loads model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [5]:
tokens = tokenizer.tokenize("The fox swims gleefully.") #TOKENIZES sequence.
tokens

['the', 'fox', 'swim', '##s', 'glee', '##fully', '.']

In [6]:
tokenizer("The fox swims gleefully") #gives variable of input_ids, token_type_ids, and attention_mask.

{'input_ids': [101, 1996, 4419, 9880, 2015, 18874, 7699, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [7]:
tokenizer.save_pretrained("directory_on_my_computer") #saving a tokenizer

('directory_on_my_computer/tokenizer_config.json',
 'directory_on_my_computer/tokenizer.json')

In [8]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint) #instantiates (new) tokenizer for checkpoint.


In [9]:
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

[1996, 4419, 9880, 2015, 18874, 7699, 1012]

In [10]:
tokenizer.convert_ids_to_tokens(ids) #gives the tokens in a list.

['the', 'fox', 'swim', '##s', 'glee', '##fully', '.']

In [11]:
tokenizer.decode(ids) #.decode() gives the original sentence.

'the fox swims gleefully.'

In [12]:
tensor_ids = torch.tensor(ids) #gives ids in tensor form, BECAUSE the model needs ids in tensor form (pytorch).
tensor_ids

tensor([ 1996,  4419,  9880,  2015, 18874,  7699,  1012])

In [13]:
tensor_ids.shape #we need this to be a 2d array, not 1d.

torch.Size([7])

In [14]:
tensor_ids_reshaped = tokenizer("The fox swims gleefully", return_tensors="pt") #return everything in tensor form.
print(tensor_ids_reshaped)
tensor_ids_reshaped = tokenizer("The fox swims gleefully", return_tensors="pt")["input_ids"] #only want input ids
print()
tensor_ids_reshaped

{'input_ids': tensor([[  101,  1996,  4419,  9880,  2015, 18874,  7699,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}



tensor([[  101,  1996,  4419,  9880,  2015, 18874,  7699,   102]])

In [15]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [17]:
model(tensor_ids_reshaped).logits

tensor([[ 0.4920,  0.4900,  0.0792, -0.3765]], grad_fn=<AddmmBackward0>)

In [18]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 892
    })
    validation: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 105
    })
    test: Dataset({
        features: ['id', 'verse_text', 'label'],
        num_rows: 104
    })
})

In [19]:
ds["train"] #the training set, with 892 training examples and 3 features (2 of which we'll use).

Dataset({
    features: ['id', 'verse_text', 'label'],
    num_rows: 892
})

In [20]:
ds["train"].features

{'id': Value('int32'),
 'verse_text': Value('string'),
 'label': ClassLabel(names=['negative', 'positive', 'no_impact', 'mixed'])}

In [24]:
print(ds['train']['verse_text'][600])
print()

label_names = ds['train'].features['label'].names
print(f"Effect: {label_names[ds["train"]['label'][600]]}")
#to get the text instead of 2.
#-use .names, and the index is the label associated w/ the index 600.

the china dustless, the keen knife-blades bright,

Effect: positive


In [27]:
print(tokenizer(ds["train"]["verse_text"][10])) #tokenizes the phrase for the 10th example of the 'verse_text' section of the training dataset.
tokenizer.tokenize(ds["train"]["verse_text"][10])

{'input_ids': [101, 1996, 2655, 1005, 1055, 2062, 13661, 2043, 2002, 19147, 4030, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


['the',
 'call',
 "'",
 's',
 'more',
 'urgent',
 'when',
 'he',
 'journeys',
 'slow',
 '.']

we must tokenize each sentence in 'verse_text', for each dataset.

In [28]:
def tokenize_function(example): #this function is mapped to the dataset, so example = dataset.
    return tokenizer(example["verse_text"], truncation=True)
    #only add another example[], as a second sentence to tokenize/input alongside the 1st sentence.

tokenized_datasets = ds.map(tokenize_function, batched=True) #THE ENTIRE DATASET tokenized

Map:   0%|          | 0/892 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

In [29]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) #use to give padding to each batch when needed. aka dynamic padding.

In [30]:
tokenized_datasets #added input_ids, token_type_ids, and attention_mask for each training & test ex.

DatasetDict({
    train: Dataset({
        features: ['id', 'verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 892
    })
    validation: Dataset({
        features: ['id', 'verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 105
    })
    test: Dataset({
        features: ['id', 'verse_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 104
    })
})

In [31]:
samples=tokenized_datasets["train"][:8] #gets 8 training examples with all features.

In [32]:
samples = {k: v for k, v in samples.items() if k not in ["id", "verse_text"]}
#remove id & verse_text
batch = data_collator(samples)

Transformer models (like BERT) are designed to receive numerical inputs such as input_ids, token_type_ids, and attention_mask. **hey don't directly process the raw text or metadata.**

DataCollatorWithpadding is also designed to handle input_ids, token_type_ids, attention_mask.  Not verse_text, or id.

In [33]:
{k: v.shape for k, v in batch.items()} #iterate through batch (dict).
#for each item in the batch (v in batch.items()), get the shape (v.shape),
#which creates the dictionary as each value (k) is v.shape, to the key (k, orig. item name)
#k, v = feature, shape

#8 is the batch size,
#26 is the sequence length (max. length for any tokenized sequence in the batch)

{'input_ids': torch.Size([8, 26]),
 'token_type_ids': torch.Size([8, 26]),
 'attention_mask': torch.Size([8, 26]),
 'labels': torch.Size([8])}

data_collator dynamically pads tokenized sequences, for each batch.

datacollatorwithPadding pads sequences within each batch, to length of longest sequence per batch.  batch is defined by the specific training exs. you choose (batch = data_collator(samples)).

it also converts the samples into tensors for model input.

### fine tuning

The first step before we can define our Trainer is to define a TrainingArguments class that will contain all the hyperparameters the Trainer will use for training and evaluation.

The only argument you have to provide is a directory where the trained model will be saved, as well as the checkpoints along the way.

In [34]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")
#push-to-hub = True as a parameter to automatically upload model to the hub.

In [35]:
#define model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Multiclass Classification: In the case of the dataset we're working with, there are four possible relationships between a verse_text and and it's effect: 'no impact', 'positive', 'negative', and 'mixed'. Therefore, num_labels must be set to 4.

In [36]:
tokenized_datasets.shape

{'train': (892, 6), 'validation': (105, 6), 'test': (104, 6)}

In [37]:
print(model.get_input_embeddings().weight.shape[0])  # model's vocab size
print(len(tokenizer))
#if they aren't the same, there's an error.

30522
30522


In [38]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

#important: to avoid cuda error, must restart session > num_labels must be 4, not 2 for all models, even the ones replaced.

When you pass a tokenizer as the processing_class, the default data_collator used by the Trainer will be a DataCollatorWithPadding. You can skip the data_collator=data_collator line in this case

In [39]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=336, training_loss=0.5329826899937221, metrics={'train_runtime': 31.5472, 'train_samples_per_second': 84.825, 'train_steps_per_second': 10.651, 'total_flos': 23879812635744.0, 'train_loss': 0.5329826899937221, 'epoch': 3.0})

### evaluation (making a compute_metrics() func. for the next time we train)

the function must take an EvalPrediction object (which is a named tuple with a predictions field and a label_ids field) and will return a dictionary mapping strings to floats (the strings being the names of the metrics returned, and the floats their values). To get some predictions from our model, we can use the Trainer.predict().

In [45]:
predictions.metrics #for analyzing

{'test_loss': 0.5431873202323914,
 'test_runtime': 0.3596,
 'test_samples_per_second': 292.008,
 'test_steps_per_second': 38.934}

In [53]:
predictions = trainer.predict(tokenized_datasets["validation"]) #get predictions using validation dataset (since we're evaluating)
print(predictions.predictions.shape, predictions.label_ids.shape)


(105, 4) (105,)
[2 1 2 0 2 0 1 0 2 2 2 0 2 2 2 2 2 2 2 2 2 2 1 2 2 2 2 2 2 2 1 2 2 2 2 2 2
 2 2 2 2 1 1 2 2 2 1 2 2 2 0 2 0 2 2 2 2 0 0 2 2 2 0 1 2 1 1 1 2 2 0 0 1 0
 2 2 2 2 1 0 1 2 2 0 0 2 2 0 2 2 2 0 2 0 2 1 2 1 2 2 2 2 2 1 0]


105 represents each logit for each element we passed into predict(), in the validation dataset.

In [46]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1) #gets index with largest value on the 2nd axis.

In [50]:
!pip install evaluate
import evaluate

In [52]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
#load-in metrics separately.

accuracy_result = accuracy_metric.compute(predictions=preds, references=predictions.label_ids)
f1_result = f1_metric.compute(predictions=preds, references=predictions.label_ids, average="macro")
#save metrics to values.
#.compute() takes predictions, references (the label_ids that compares against predicted labels, preds))
#do macro for f1, precision, and recall, and micro is typically for accuracy.

print("Accuracy:", accuracy_result)
print("F1-Score (macro):", f1_result)

Accuracy: {'accuracy': 0.8761904761904762}
F1-Score (macro): {'f1': 0.8109699536196814}


we can combine all of these steps into a function called compute_metrics:

In [57]:
def compute_metrics(eval_preds):
    acc_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")


    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    #use labels, not predictions.label_ids
    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    # Trainer expects a single dictionary of metrics
    return {**acc, **f1}

When you pass this function to Trainer(compute_metrics=compute_metrics, ...), it'll log both eval_accuracy and eval_f1 during evaluation, since the Trainer prefixes each key in the returned dict with eval_.

So, you can either do return metric.compute(...), for a dict for multiple metrics.



---


re-define the trainer, to see it report metrics after every epoch:

In [58]:
training_args = TrainingArguments("test-trainer", eval_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=4)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics, #here.
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [59]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.421665,0.885714,0.825275
2,No log,0.641982,0.809524,0.725682
3,No log,0.558396,0.876190,0.613917


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=336, training_loss=0.4968946093604678, metrics={'train_runtime': 53.0561, 'train_samples_per_second': 50.437, 'train_steps_per_second': 6.333, 'total_flos': 23879812635744.0, 'train_loss': 0.4968946093604678, 'epoch': 3.0})

Here, we trained the model with an average accuracy of 86%, and an f1-score of 0.72!